# Pipeline Benchmark: Original vs Current Clip Processing

A head-to-head run of the original pipeline's clip processing ([`original_pipeline.ipynb`](original_pipeline.ipynb), "old" below) against the current scraper's ([`twitch_vod_scraper.ipynb`](../twitch_vod_scraper.ipynb), "new") **on the same clips from the same VOD**, so every performance claim is a measured number, not an estimate.

### What happens, in order
1. **Download** the VOD (yt-dlp) and its chat (the GQL scraper). TwitchDownloaderCLI, which the original pipeline used, segfaults on current Colab, so both methods share the current downloader.
2. **Pick clips** with the original chat parser: BTTV-only emotes, whole-fragment matching, and at least 40 emotes per 15-second window. Both methods then process this exact clip list.
3. **Process every clip twice, back to back.** Once with the old code into `dataset_<VOD>/`, and once with the new code into `dataset_<VOD>_new/`. The order alternates each clip (old first, then new first) so neither method always gets a warmer disk cache. Each method has its own video reader, so they don't interfere. Every target, video and audio step is timed for both.
4. **Export both**: the old way (zip the whole folder, `full_vod.mp4` included) and the new way (zip only the clip triplets).
5. **Summarise** into plain headline numbers.

### What differs between the two methods
| Step | Old | New |
|---|---|---|
| Target | emote vector | same maths |
| Video | reads the VOD sequentially without seeking, so each clip's frames come from earlier in the VOD than its chat window | seeks to the clip's start frame |
| Audio | `ffmpeg -i VOD -ss T` decodes from 0s to T on every clip | `ffmpeg -ss T -i VOD` jumps straight to T |
| Export | zips the whole output folder, VOD included | zips clips and emote mapping only |

Both methods run their notebooks' processing code as written. The only changes are timers, logging, a readability check on the VOD file, and a whole-clip resume check in place of per-file existence checks. Download and chat timings are shared and say nothing about the original pipeline.

### Outputs (written incrementally to Google Drive, so a cleared Colab output loses nothing)
| File | Contents |
|---|---|
| `events.log` | Timestamped log of everything printed |
| `environment.json` | CPU, RAM, ffmpeg/OpenCV versions, VOD file size |
| `stages.json` | Wall time and details for every stage |
| `clips.csv` | One row per clip: old and new target/video/audio times, VOD position, where each method's frames actually came from |
| `per_vod_hour.csv` | Old vs new mean per-clip times, bucketed by hour into the VOD |
| `summary.json` | Every derived number, plus headline statements |

### Running it
1. Add the Colab secrets `CLIENT_ID`, `AUTHORIZATION`, `CLIENT_INTEGRITY` and `DEVICE_ID`, exactly as for the scraper.
2. Set `VOD_ID`. Use a **long** VOD (5h+), because the old audio cost grows with VOD length.
3. Run all. A full run can take hours, almost all of it the old method. To shorten it, set `CLIP_STRIDE` to process every Nth clip; this keeps clips spread across the VOD, and both totals are scaled up identically.
4. If the runtime disconnects (but is not reset), set `RESUME_LOG_DIR` to the printed log folder and re-run. Clips with both outputs complete are skipped, and a half-finished clip is redone from scratch.

## Results of the saved run

The outputs below are from one Colab run (8 CPUs) on VOD `1594760722` (6.6 h). Processing was stopped manually after **443 of 1,225 clips** (0.6 h to 2.5 h into the VOD), because the old method's per-clip time keeps growing and a full run would have taken many hours. The `KeyboardInterrupt` in the pipeline cell is that stop; the analysis cells use every clip that finished.

Raw logs from this run are in [`results/`](results/), and the write-up is in [`README.md`](README.md).

In [5]:
# ==========================================
# ⚙️ CONFIGURATION
# ==========================================
VOD_ID = "1594760722"   # ← use a long VOD; the audio bug scales with VOD length
VOD_URL = f"https://www.twitch.tv/videos/{VOD_ID}"
OUTPUT_FOLDER = f"./dataset_{VOD_ID}"          # old method output
OUTPUT_FOLDER_NEW = f"./dataset_{VOD_ID}_new"  # new method output

# --- Pipeline parameters (unchanged from clip_downloader_old.ipynb) ---
CLIP_DURATION = 15
MIN_EMOTES = 40         # Minimum emotes in a CLIP_DURATION window. Named MIN_CHAT_MESSAGES in the old
                        # notebook, but it has always counted emotes. Lowering it changes the old behaviour.
TARGET_RES = 256
CHAT_LAG_OFFSET = 5     # Defined but unused in the old pipeline too.

# --- Benchmark controls ---
CLIP_STRIDE = 1         # Process every Nth interesting clip. 1 = every clip.
PROGRESS_EVERY = 10     # Log a progress line every N clips.
RUN_EXPORT_BENCHMARK = True

DRIVE_LOG_FOLDER = "/content/drive/MyDrive/twitch_emote_benchmarks"
RESUME_LOG_DIR = None   # e.g. "/content/drive/MyDrive/twitch_emote_benchmarks/old_vs_new_<VOD>_<run>"

In [6]:
!pip install -q yt-dlp curl-cffi

## Logging and environment

In [7]:
import csv
import json
import os
import platform
import shutil
import subprocess
import time
import zipfile
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import tqdm
import yt_dlp
from curl_cffi import requests
from google.colab import drive, userdata

drive.mount('/content/drive')

if RESUME_LOG_DIR:
    LOG_DIR = Path(RESUME_LOG_DIR)
else:
    LOG_DIR = Path(DRIVE_LOG_FOLDER) / f"old_vs_new_{VOD_ID}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
LOG_DIR.mkdir(parents=True, exist_ok=True)

EVENTS_LOG = LOG_DIR / "events.log"
ENV_JSON = LOG_DIR / "environment.json"
STAGES_JSON = LOG_DIR / "stages.json"
CLIPS_CSV = LOG_DIR / "clips.csv"
SUMMARY_JSON = LOG_DIR / "summary.json"


def read_json(path):
    if path.exists():
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return {}


def write_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)


STAGES = read_json(STAGES_JSON)
SUMMARY = read_json(SUMMARY_JSON)
RUN_STATE = {}


def fmt_duration(seconds):
    seconds = float(seconds)
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{int(hours)}h {int(minutes):02d}m {secs:04.1f}s"
    if minutes:
        return f"{int(minutes)}m {secs:04.1f}s"
    return f"{secs:.2f}s"


def log(msg):
    line = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line)
    with open(EVENTS_LOG, "a", encoding="utf-8") as f:
        f.write(line + "\n")


@contextmanager
def stage(name):
    """Times a pipeline stage. The body can add details to the yielded dict."""
    details = {}
    started = datetime.now()
    log(f"▶ {name}")
    t0 = time.perf_counter()
    status = "ok"
    try:
        yield details
    except KeyboardInterrupt:
        status = "interrupted"
        raise
    except BaseException:
        status = "failed"
        raise
    finally:
        elapsed = time.perf_counter() - t0
        # A resumed run keeps earlier sessions' timings instead of overwriting them.
        key, session = name, 1
        while key in STAGES:
            session += 1
            key = f"{name} (session {session})"
        details.update(seconds=round(elapsed, 3), human=fmt_duration(elapsed), status=status,
                       started=started.isoformat(timespec="seconds"))
        STAGES[key] = dict(details)
        write_json(STAGES_JSON, STAGES)
        log(f"■ {key}: {fmt_duration(elapsed)} ({status})")


def capture_environment():
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg is not on PATH. Run: !apt-get install -y ffmpeg")
    mem_gb = None
    try:
        with open("/proc/meminfo") as f:
            mem_gb = round(int(f.readline().split()[1]) / 1e6, 1)
    except OSError:
        pass
    env = {
        "run_started": datetime.now().isoformat(timespec="seconds"),
        "vod_id": VOD_ID,
        "python": platform.python_version(),
        "platform": platform.platform(),
        "cpu_count": os.cpu_count(),
        "ram_gb": mem_gb,
        "ffmpeg": subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0],
        "opencv": cv2.__version__,
        "yt_dlp": yt_dlp.version.__version__,
        "config": {
            "CLIP_DURATION": CLIP_DURATION, "MIN_EMOTES": MIN_EMOTES,
            "TARGET_RES": TARGET_RES, "CLIP_STRIDE": CLIP_STRIDE,
        },
    }
    write_json(ENV_JSON, env)
    return env


# --- Credentials (same secrets as twitch_vod_scraper.ipynb) ---
MY_CLIENT_ID = userdata.get('CLIENT_ID')
MY_OAUTH = userdata.get('AUTHORIZATION')
MY_INTEGRITY = userdata.get('CLIENT_INTEGRITY')
DEVICE_ID = userdata.get('DEVICE_ID')

_missing = [name for name, val in {
    'CLIENT_ID': MY_CLIENT_ID,
    'AUTHORIZATION': MY_OAUTH,
    'CLIENT_INTEGRITY': MY_INTEGRITY,
    'DEVICE_ID': DEVICE_ID,
}.items() if not val]
if _missing:
    raise RuntimeError(
        f"Missing Colab secret(s): {', '.join(_missing)}\n"
        "Go to Tools → Secrets and add them before running."
    )

ENV = capture_environment()
log(f"Logging to {LOG_DIR}")
log(f"{ENV['cpu_count']} CPUs, {ENV['ram_gb']} GB RAM | {ENV['ffmpeg']} | OpenCV {ENV['opencv']}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[2026-09-13 22:41:57] Logging to /content/drive/MyDrive/twitch_emote_benchmarks/old_vs_new_1594760722_20260913_224155
[2026-09-13 22:41:57] 8 CPUs, 53.5 GB RAM | ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers | OpenCV 5.0.0


## Helpers

`get_bttv_emotes` is copied verbatim from the original notebook. `get_vod_metadata` and the chat scraper are the current method; the scraper additionally keeps raw fragments so the old parser can consume them.

In [8]:
# ==========================================
# 1. BTTV HELPER (verbatim from the old pipeline)
# ==========================================
def get_bttv_emotes(channel_id):
    """Fetches Global and Channel-specific BTTV emotes."""
    emotes = set()
    try:
        # 1. Global Emotes
        resp = requests.get("https://api.betterttv.net/3/cached/emotes/global")
        if resp.status_code == 200:
            for e in resp.json():
                emotes.add(e['code'])

        # 2. Channel Emotes
        if channel_id:
            resp = requests.get(f"https://api.betterttv.net/3/cached/users/twitch/{channel_id}")
            if resp.status_code == 200:
                data = resp.json()
                for e in data.get('channelEmotes', []) + data.get('sharedEmotes', []):
                    emotes.add(e['code'])
    except Exception as e:
        log(f"⚠️ BTTV Fetch Warning: {e}")

    log(f"   ✅ Loaded {len(emotes)} BTTV emote codes.")
    return emotes


# ==========================================
# 2. REPLACEMENT DOWNLOAD HELPERS (current method)
# ==========================================
def get_vod_metadata(vod_id):
    ydl_opts = {'quiet': True, 'simulate': True}

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(f"https://www.twitch.tv/videos/{vod_id}", download=False)

            duration = info.get('duration')
            streamer_name = info.get('uploader_id') or info.get('uploader')

            if streamer_name:
                resp = requests.get(f"https://decapi.me/twitch/id/{streamer_name}", timeout=5)
                if resp.status_code == 200 and "User not found" not in resp.text:
                    channel_id = resp.text.strip()
                    if channel_id and duration:
                        return channel_id, duration
                    if not duration:
                        log("⚠️ VOD duration could not be determined automatically.")
                else:
                    log(f"⚠️ DecAPI could not resolve ID for {streamer_name}.")
            else:
                log("⚠️ Streamer name not found in VOD metadata.")

    except Exception as e:
        log(f"⚠️ Error during metadata fetch: {e}")

    log("❌ Automation failed. Manual input required.")
    manual_id = input("   👉 Go to https://decapi.me/twitch/id/STREAMER_NAME in your browser and paste the number here: ")
    manual_duration = int(input("   👉 Enter the VOD duration in seconds (e.g. 16927): "))
    return str(manual_id).strip(), manual_duration


def fetch_chat_gql_raw(vod_id, max_vod_seconds):
    """The GQL scraper from twitch_vod_scraper.ipynb, keeping each message's raw
    fragments (text + emote marker) instead of flattening them, so the old
    TwitchDownloaderCLI-format parser can run unchanged."""
    url = "https://gql.twitch.tv/gql"
    headers = {
        "Client-Id": MY_CLIENT_ID,
        "Authorization": MY_OAUTH,
        "Client-Integrity": MY_INTEGRITY,
        "X-Device-Id": DEVICE_ID,
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0",
        "Content-Type": "application/json",
    }

    messages = []
    seen_messages = set()
    current_offset = 0
    stats = {"requests": 0, "retries": 0, "skipped_seconds": 0, "resumed_from_s": None, "stopped_early": None}

    # --- RESUME LOGIC ---
    save_file = f"chat_raw_cache_{vod_id}.json"
    if os.path.exists(save_file):
        try:
            with open(save_file, "r", encoding="utf-8") as f:
                messages = json.load(f)
            if messages:
                current_offset = int(messages[-1]['content_offset_seconds'])
                seen_messages = {m['id'] for m in messages}
                stats["resumed_from_s"] = current_offset
                log(f"Resuming chat from {current_offset}s ({len(messages)} messages cached).")
        except Exception as e:
            log(f"⚠️ Could not load chat cache: {e}. Starting from scratch.")
            messages = []
            current_offset = 0

    MAX_RETRIES = 3
    retry_count = 0
    cursor = None

    with tqdm.tqdm(total=max_vod_seconds, initial=int(current_offset), desc="Scraping Timeline") as pbar:
        while current_offset <= max_vod_seconds:
            variables = {"videoID": vod_id, "contentOffsetSeconds": int(current_offset)}
            if cursor:
                variables["cursor"] = cursor

            query = [{
                "operationName": "VideoCommentsByOffsetOrCursor",
                "variables": variables,
                "extensions": {
                    "persistedQuery": {
                        "version": 1,
                        "sha256Hash": "b70a3591ff0f4e0313d126c6a1502d79a1c02baebb288227c582044aa76adf6a"
                    }
                }
            }]

            try:
                resp = requests.post(url, json=query, headers=headers, timeout=10)
                stats["requests"] += 1

                if resp.status_code != 200:
                    stats["stopped_early"] = f"HTTP {resp.status_code} at {current_offset}s"
                    log(f"❌ HTTP {resp.status_code} at {current_offset}s. Check that your client integrity token is valid.")
                    break

                res_data = resp.json()[0]

                if 'errors' in res_data:
                    error_msg = res_data['errors'][0]['message']
                    if "service error" in error_msg.lower() or "timeout" in error_msg.lower():
                        if retry_count < MAX_RETRIES:
                            retry_count += 1
                            stats["retries"] += 1
                            time.sleep(1.5)
                            continue
                        current_offset += 1
                        cursor = None
                        pbar.update(1)
                        retry_count = 0
                        stats["skipped_seconds"] += 1
                        continue
                    stats["stopped_early"] = f"Twitch error at {current_offset}s: {error_msg}"
                    log(f"❌ Unrecoverable Twitch error at {current_offset}s: {error_msg}")
                    break

                retry_count = 0

                comments_data = ((res_data.get('data') or {}).get('video') or {}).get('comments') or {}
                edges = comments_data.get('edges', [])
                has_next_page = comments_data.get('pageInfo', {}).get('hasNextPage', False)

                if not edges:
                    current_offset += 1
                    cursor = None
                    pbar.update(1)
                    continue

                for edge in edges:
                    node = edge['node']
                    msg_id = node.get('id')
                    if msg_id in seen_messages:
                        continue
                    seen_messages.add(msg_id)

                    fragments = []
                    for frag in (node.get('message') or {}).get('fragments') or []:
                        emote = frag.get('emote')
                        fragments.append({
                            "text": frag.get('text', ''),
                            # TwitchDownloaderCLI marks native emotes with an 'emoticon' object
                            "emoticon": {"emoticon_id": emote.get('emoteID')} if emote else None,
                        })

                    messages.append({
                        "id": msg_id,
                        "content_offset_seconds": node.get('contentOffsetSeconds', current_offset),
                        "fragments": fragments,
                    })

                # --- PAGINATION LOGIC ---
                next_offset = int(edges[-1]['node'].get('contentOffsetSeconds', current_offset))
                progress_made = next_offset - int(current_offset)
                if progress_made > 0:
                    pbar.update(progress_made)
                    current_offset = next_offset

                if has_next_page:
                    cursor = edges[-1].get('cursor')
                else:
                    cursor = None
                    if progress_made <= 0:
                        current_offset += 1
                        pbar.update(1)

                time.sleep(0.2)

                # --- PERIODIC SAVE ---
                if stats["requests"] % 50 == 0:
                    with open(save_file, "w", encoding="utf-8") as f:
                        json.dump(messages, f)

            except Exception as e:
                if retry_count < MAX_RETRIES:
                    retry_count += 1
                    stats["retries"] += 1
                    log(f"⚠️ Network error at {current_offset}s: {e}. Retrying ({retry_count}/{MAX_RETRIES})...")
                    time.sleep(3)
                    continue
                stats["stopped_early"] = f"Network error at {current_offset}s: {e}"
                log(f"❌ Network error unrecoverable after {MAX_RETRIES} retries. Stopping.")
                break

    with open(save_file, "w", encoding="utf-8") as f:
        json.dump(messages, f)

    stats["messages"] = len(messages)
    stats["last_offset_s"] = messages[-1]['content_offset_seconds'] if messages else 0
    return messages, stats


def to_twitchdownloader_schema(messages, channel_id):
    """Shapes GQL messages like TwitchDownloaderCLI's chat.json — only the fields the old parser reads."""
    return {
        "streamer": {"id": channel_id},
        "comments": [
            {
                "_id": m["id"],
                "content_offset_seconds": m["content_offset_seconds"],
                "message": {"fragments": m["fragments"]},
            }
            for m in messages
        ],
    }

## Pipeline: shared download, then old and new processing on every clip

Steps A and B are the original notebook's code (with the download swapped). Step C holds both methods' clip processing, each copied from its notebook. Comments marked `# [old]` point out behaviour that matters for interpreting the timings.

In [9]:
CLIP_FIELDS = [
    "clip_index", "clip_id", "start_time_s", "start_time_h", "skipped", "first",
    "old_target_s", "old_video_s", "old_audio_s", "old_total_s",
    "new_target_s", "new_video_s", "new_audio_s", "new_total_s",
    "old_frames_written", "new_frames_read",
    "old_video_source_start_s", "old_video_source_end_s", "new_video_source_end_s",
    "elapsed_s", "wall_clock",
]

def run_pipeline():
    # Create Directories
    Path(OUTPUT_FOLDER).mkdir(parents=True, exist_ok=True)
    video_dir = Path(OUTPUT_FOLDER) / "video"
    audio_dir = Path(OUTPUT_FOLDER) / "audio"
    target_dir = Path(OUTPUT_FOLDER) / "target"

    for d in [video_dir, audio_dir, target_dir]:
        d.mkdir(parents=True, exist_ok=True)

    chat_path = f"{OUTPUT_FOLDER}/chat.json"
    video_path = f"{OUTPUT_FOLDER}/full_vod.mp4"
    RUN_STATE.update(video_path=video_path, chat_path=chat_path,
                     video_dir=video_dir, audio_dir=audio_dir, target_dir=target_dir)

    # --- STEP A: DOWNLOAD (TwitchDownloaderCLI replaced with the current method) ---
    with stage("vod_metadata") as info:
        channel_id, vod_duration = get_vod_metadata(VOD_ID)
        info.update(channel_id=channel_id, vod_duration_s=vod_duration,
                    vod_duration_h=round(vod_duration / 3600, 2))
    RUN_STATE["vod_duration"] = vod_duration

    with stage("download_chat [new method]") as info:
        if os.path.exists(chat_path):
            info["skipped"] = True
            log("chat.json already exists, skipping download (this stage's time is not meaningful).")
        else:
            messages, chat_stats = fetch_chat_gql_raw(VOD_ID, vod_duration)
            info.update(chat_stats)
            if not messages:
                raise RuntimeError("No chat messages retrieved. Verify the Colab secrets are valid.")
            coverage = chat_stats["last_offset_s"] / vod_duration
            info["coverage_pct"] = round(100 * coverage, 1)
            if coverage < 0.9:
                log(f"⚠️ WARNING: chat only covers {100 * coverage:.1f}% of the VOD "
                    f"({chat_stats['stopped_early']}). Clip counts will be low; consider refreshing "
                    "the integrity token, deleting dataset_*/chat.json and re-running (the cache resumes).")
            with open(chat_path, "w", encoding="utf-8") as f:
                json.dump(to_twitchdownloader_schema(messages, channel_id), f)

    with stage("download_video [new method]") as video_info:
        if os.path.exists(video_path):
            video_info["skipped"] = True
            log("Video already downloaded, skipping (this stage's time is not meaningful).")
        else:
            ydl_opts = {
                'format': 'best[height<=360]/best',
                'outtmpl': video_path,
                'merge_output_format': 'mp4',
                'concurrent_fragment_downloads': 5,
                'quiet': False,
            }
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([VOD_URL])
        video_info["file_gb"] = round(os.path.getsize(video_path) / 1e9, 3)
    ENV["vod_file_gb"] = video_info["file_gb"]
    write_json(ENV_JSON, ENV)

    # --- STEP B: ANALYZE CHAT (verbatim) ---
    with stage("analyze_chat") as info:
        with open(chat_path, 'r', encoding='utf-8') as f:
            chat_data = json.load(f)

        streamer_id = chat_data.get('streamer', {}).get('id')
        bttv_emotes = get_bttv_emotes(streamer_id)

        buckets = {}

        for comment in chat_data['comments']:
            offset = int(comment['content_offset_seconds'])
            window_start = (offset // CLIP_DURATION) * CLIP_DURATION

            if window_start not in buckets: buckets[window_start] = {}

            msg_obj = comment.get('message', {})
            fragments = msg_obj.get('fragments', [])

            for frag in fragments:
                text = frag.get('text', '').strip()

                # Check 1: Twitch Native Emote (has 'emoticon' object)
                if frag.get('emoticon') is not None:
                    buckets[window_start][text] = buckets[window_start].get(text, 0) + 1

                # Check 2: BTTV Emote (Text match)
                # [old] matches the whole fragment, so "lol KEKW" counts nothing
                elif text in bttv_emotes:
                    buckets[window_start][text] = buckets[window_start].get(text, 0) + 1

        interesting_starts = []
        for t, dist in buckets.items():
            if sum(dist.values()) >= MIN_EMOTES:
                interesting_starts.append(t)

        interesting_starts.sort()
        log(f"✅ Found {len(interesting_starts)} clips where emote count >= {MIN_EMOTES}.")

        all_unique_emotes = set()
        for t, dist in buckets.items():
            for emote_code in dist.keys():
                all_unique_emotes.add(emote_code)

        sorted_emotes = sorted(list(all_unique_emotes))
        emote_to_index = {emote: i for i, emote in enumerate(sorted_emotes)}

        emote_mapping_path = Path(OUTPUT_FOLDER) / 'emote_mapping.json'
        with open(emote_mapping_path, 'w') as f:
            json.dump(emote_to_index, f)
        log(f"Total unique emotes identified: {len(sorted_emotes)}")

        info.update(comments=len(chat_data['comments']), windows_with_chat=len(buckets),
                    bttv_codes=len(bttv_emotes), interesting_clips=len(interesting_starts),
                    unique_emotes=len(sorted_emotes))

    if not interesting_starts:
        raise RuntimeError(f"No window reached {MIN_EMOTES} emotes. Pick a busier VOD.")

    selected_starts = interesting_starts[::CLIP_STRIDE]
    RUN_STATE.update(interesting_starts=interesting_starts, selected_starts=selected_starts)
    log(f"Processing {len(selected_starts)} of {len(interesting_starts)} clips (CLIP_STRIDE={CLIP_STRIDE}) with both "
        f"methods, spanning {selected_starts[0] / 3600:.2f}h to {selected_starts[-1] / 3600:.2f}h of the VOD.")

    new_video_dir = Path(OUTPUT_FOLDER_NEW) / "video"
    new_audio_dir = Path(OUTPUT_FOLDER_NEW) / "audio"
    new_target_dir = Path(OUTPUT_FOLDER_NEW) / "target"
    for d in [new_video_dir, new_audio_dir, new_target_dir]:
        d.mkdir(parents=True, exist_ok=True)
    shutil.copy(emote_mapping_path, Path(OUTPUT_FOLDER_NEW) / 'emote_mapping.json')

    FRAMES_PER_CLIP = 64

    def old_paths(clip_id):
        return video_dir / f"{clip_id}.mp4", audio_dir / f"{clip_id}.mp3", target_dir / f"{clip_id}.json"

    def new_paths(clip_id):
        return new_video_dir / f"{clip_id}.mp4", new_audio_dir / f"{clip_id}.mp3", new_target_dir / f"{clip_id}.json"

    # ------------------------------------------------------------------
    # OLD method: clip_downloader_old.ipynb, verbatim apart from timers
    # ------------------------------------------------------------------
    def process_clip_old(cap, geometry, start_time, clip_id):
        original_fps, x_start, y_start, x_end, y_end = geometry
        output_clip_fps = FRAMES_PER_CLIP / CLIP_DURATION
        vid_file, aud_file, tgt_file = old_paths(clip_id)
        t = {}

        # 1. Save Target JSON (The Normalized Emote Frequency Vector)
        t0 = time.perf_counter()
        emote_vector = np.zeros(len(sorted_emotes))
        for emote, count in buckets[start_time].items():
            if emote in emote_to_index:
                emote_vector[emote_to_index[emote]] = count

        vector_sum = emote_vector.sum()
        if vector_sum > 0:
            emote_vector = emote_vector / vector_sum

        with open(tgt_file, 'w') as f:
            json.dump(emote_vector.tolist(), f)
        t["target_s"] = time.perf_counter() - t0

        # 2. Process Video (Middle Crop)
        t0 = time.perf_counter()
        t["video_source_start_s"] = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000
        writer = cv2.VideoWriter(str(vid_file), cv2.VideoWriter_fourcc(*'mp4v'), output_clip_fps, (TARGET_RES, TARGET_RES))
        start_pos = int(start_time * original_fps)  # [old] computed but never used: the capture never seeks
        frames_captured = 0
        current_frame_idx = 0
        total_source_frames = int(CLIP_DURATION * original_fps)

        while frames_captured < FRAMES_PER_CLIP:
            ret, frame = cap.read()
            if not ret: break

            target_frame_for_capture = int(frames_captured * (total_source_frames / FRAMES_PER_CLIP))

            if current_frame_idx == target_frame_for_capture:
                crop = frame[y_start:y_end, x_start:x_end]
                final = cv2.resize(crop, (TARGET_RES, TARGET_RES))
                writer.write(final)
                frames_captured += 1

            current_frame_idx += 1

        writer.release()
        t["frames_written"] = frames_captured
        t["video_source_end_s"] = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000
        t["video_s"] = time.perf_counter() - t0

        # 3. Extract Audio
        t0 = time.perf_counter()
        # [old] -ss after -i: output seeking, decodes from 0s to start_time on every call
        subprocess.run([
            "ffmpeg", "-y", "-i", video_path, "-ss", str(start_time), "-t", str(CLIP_DURATION),
            "-vn", "-acodec", "libmp3lame", str(aud_file)
        ], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        t["audio_s"] = time.perf_counter() - t0
        return t

    # ------------------------------------------------------------------
    # NEW method: twitch_vod_scraper.ipynb, verbatim apart from timers
    # ------------------------------------------------------------------
    def process_clip_new(cap, geometry, start_time, clip_id):
        original_fps, x, y, s = geometry
        out_fps = FRAMES_PER_CLIP / CLIP_DURATION
        vid_f, aud_f, tgt_f = new_paths(clip_id)
        t = {}

        # 1. Target Vector
        t0 = time.perf_counter()
        emote_vector = np.zeros(len(all_unique_emotes))
        for emote, count in buckets[start_time].items():
            if emote != '_total_messages':
                emote_vector[emote_to_index[emote]] = count

        if emote_vector.sum() > 0:
            emote_vector /= emote_vector.sum()

        with open(tgt_f, 'w') as f:
            json.dump(emote_vector.tolist(), f)
        t["target_s"] = time.perf_counter() - t0

        # 2. Video Crop & Resample
        t0 = time.perf_counter()
        start_frame = int(start_time * original_fps)
        end_frame = int((start_time + CLIP_DURATION) * original_fps)
        total_frames = end_frame - start_frame

        sample_indices = set(np.linspace(0, total_frames, FRAMES_PER_CLIP, endpoint=False, dtype=int))

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        frames_to_write = []

        for rel_idx in range(total_frames):
            ret, frame = cap.read()
            if not ret:
                break
            if rel_idx in sample_indices:
                crop = frame[y:y+s, x:x+s]
                resized = cv2.resize(crop, (TARGET_RES, TARGET_RES))
                frames_to_write.append(resized)
        t["frames_read"] = len(frames_to_write)
        t["video_source_end_s"] = cap.get(cv2.CAP_PROP_POS_MSEC) / 1000

        if len(frames_to_write) == 0:
            frames_to_write = [np.zeros((TARGET_RES, TARGET_RES, 3), dtype=np.uint8)] * FRAMES_PER_CLIP
        elif len(frames_to_write) < FRAMES_PER_CLIP:
            last_frame = frames_to_write[-1]
            padding = [last_frame] * (FRAMES_PER_CLIP - len(frames_to_write))
            frames_to_write.extend(padding)
        elif len(frames_to_write) > FRAMES_PER_CLIP:
            frames_to_write = frames_to_write[:FRAMES_PER_CLIP]

        writer = cv2.VideoWriter(str(vid_f), cv2.VideoWriter_fourcc(*'mp4v'), out_fps, (TARGET_RES, TARGET_RES))
        for f in frames_to_write:
            writer.write(f)
        writer.release()
        t["video_s"] = time.perf_counter() - t0

        # 3. Audio Extraction
        t0 = time.perf_counter()
        try:
            subprocess.run([
                "ffmpeg", "-y", "-ss", str(start_time), "-t", str(CLIP_DURATION),
                "-i", video_path, "-vn", "-acodec", "libmp3lame", str(aud_f)
            ], check=True, capture_output=True)
        except subprocess.CalledProcessError as e:
            log(f"⚠️ Audio extraction failed for clip at {start_time}s: {e}")
        t["audio_s"] = time.perf_counter() - t0
        return t

    # --- STEP C: PROCESS EVERY CLIP WITH BOTH METHODS ---
    csv_exists = CLIPS_CSV.exists()
    with stage("process_clips [old + new, interleaved]") as info, \
            open(CLIPS_CSV, "a", newline="", encoding="utf-8") as csv_f:
        clip_log = csv.DictWriter(csv_f, fieldnames=CLIP_FIELDS)
        if not csv_exists:
            clip_log.writeheader()

        # Separate readers: the old method depends on where its reader last stopped.
        cap_old = cv2.VideoCapture(video_path)
        cap_new = cv2.VideoCapture(video_path)
        try:
            if not (cap_old.isOpened() and cap_new.isOpened()):
                raise RuntimeError(f"Could not open '{video_path}'. Delete it and re-run the download.")

            # Old geometry (verbatim)
            original_fps = cap_old.get(cv2.CAP_PROP_FPS)
            original_width = int(cap_old.get(cv2.CAP_PROP_FRAME_WIDTH))
            original_height = int(cap_old.get(cv2.CAP_PROP_FRAME_HEIGHT))
            square_side = min(original_width, original_height)
            x_start = (original_width - square_side) // 2
            y_start = (original_height - square_side) // 2
            old_geometry = (original_fps, x_start, y_start, x_start + square_side, y_start + square_side)

            # New geometry (verbatim)
            new_fps = cap_new.get(cv2.CAP_PROP_FPS)
            orig_w = int(cap_new.get(cv2.CAP_PROP_FRAME_WIDTH))
            orig_h = int(cap_new.get(cv2.CAP_PROP_FRAME_HEIGHT))
            if new_fps <= 0 or orig_w <= 0 or orig_h <= 0:
                raise RuntimeError(f"Unusable video properties: fps={new_fps}, dimensions={orig_w}x{orig_h}.")
            s = min(orig_w, orig_h)
            x, y = (orig_w - s) // 2, (orig_h - s) // 2
            new_geometry = (new_fps, x, y, s)
            info.update(source_fps=original_fps, source_resolution=f"{original_width}x{original_height}")

            totals = {f"{v}_{k}": 0.0 for v in ("old", "new") for k in ("target_s", "video_s", "audio_s")}
            measured = skipped = 0
            loop_t0 = time.perf_counter()

            for clip_counter, start_time in enumerate(tqdm.tqdm(selected_starts, desc="Processing clips (old + new)")):
                clip_id = f"clip_{VOD_ID}_{clip_counter:05d}"
                outputs = [*old_paths(clip_id), *new_paths(clip_id)]
                row = {"clip_index": clip_counter, "clip_id": clip_id,
                       "start_time_s": start_time, "start_time_h": round(start_time / 3600, 4)}

                if all(p.exists() for p in outputs):
                    row.update(skipped=True, wall_clock=datetime.now().isoformat(timespec="seconds"))
                    clip_log.writerow(row)
                    skipped += 1
                    continue
                for p in outputs:  # clear partial output from an interrupted clip so both methods start clean
                    p.unlink(missing_ok=True)

                order = ("old", "new") if clip_counter % 2 == 0 else ("new", "old")
                row.update(skipped=False, first=order[0])
                for variant in order:
                    if variant == "old":
                        result = process_clip_old(cap_old, old_geometry, start_time, clip_id)
                    else:
                        result = process_clip_new(cap_new, new_geometry, start_time, clip_id)
                    for key, value in result.items():
                        row[f"{variant}_{key}"] = round(value, 4) if isinstance(value, float) else value
                    row[f"{variant}_total_s"] = round(result["target_s"] + result["video_s"] + result["audio_s"], 4)

                row["elapsed_s"] = round(time.perf_counter() - loop_t0, 2)
                row["wall_clock"] = datetime.now().isoformat(timespec="seconds")
                clip_log.writerow(row)
                csv_f.flush()

                measured += 1
                for key in totals:
                    totals[key] += row[key]
                if measured % PROGRESS_EVERY == 0 or clip_counter == len(selected_starts) - 1:
                    old_sum = sum(v for k, v in totals.items() if k.startswith("old_"))
                    new_sum = sum(v for k, v in totals.items() if k.startswith("new_"))
                    log(f"clip {clip_counter + 1}/{len(selected_starts)} | VOD pos {start_time / 3600:.2f}h | "
                        f"old: audio {row['old_audio_s']:.2f}s video {row['old_video_s']:.2f}s | "
                        f"new: audio {row['new_audio_s']:.2f}s video {row['new_video_s']:.2f}s | "
                        f"so far old {fmt_duration(old_sum)} vs new {fmt_duration(new_sum)}")
        finally:
            cap_old.release()
            cap_new.release()

        info.update(clips_measured=measured, clips_skipped=skipped,
                    **{f"total_{k}": round(v, 2) for k, v in totals.items()})

    log(f"🎉 Done! {measured} clips processed with both methods "
        f"(old → {OUTPUT_FOLDER}, new → {OUTPUT_FOLDER_NEW}).")

with stage("pipeline_total"):
    run_pipeline()

[2026-09-13 22:41:57] ▶ pipeline_total
[2026-09-13 22:41:57] ▶ vod_metadata
[2026-09-13 22:41:58] ■ vod_metadata: 1.68s (ok)
[2026-09-13 22:41:58] ▶ download_chat [new method]


Scraping Timeline: 23884it [16:51, 23.61it/s]                           


[2026-09-13 22:58:56] ■ download_chat [new method]: 16m 58.1s (ok)
[2026-09-13 22:58:56] ▶ download_video [new method]
[twitch:vod] Extracting URL: https://www.twitch.tv/videos/1594760722
[twitch:vod] 1594760722: Downloading stream metadata GraphQL
[twitch:vod] 1594760722: Downloading video access token GraphQL
[twitch:vod] 1594760722: Downloading m3u8 information
[twitch:vod] 1594760722: Downloading storyboard metadata JSON
[info] v1594760722: Downloading 1 format(s): 360p
[hlsnative] Downloading m3u8 manifest
[hlsnative] Total fragments: 2006
[download] Destination: ./dataset_1594760722/full_vod.mp4
[download] 100% of    1.95GiB in 00:03:25 at 9.72MiB/s                     
[FixupM3u8] Fixing MPEG-TS in MP4 container of "./dataset_1594760722/full_vod.mp4"
[2026-09-13 23:02:49] ■ download_video [new method]: 3m 52.9s (ok)
[2026-09-13 23:02:49] ▶ analyze_chat
[2026-09-13 23:02:50]    ✅ Loaded 89 BTTV emote codes.
[2026-09-13 23:02:50] ✅ Found 1225 clips where emote count >= 40.
[2026-0

Processing clips (old + new):   1%|          | 10/1225 [01:15<2:34:05,  7.61s/it]

[2026-09-13 23:04:07] clip 10/1225 | VOD pos 0.65h | old: audio 6.09s video 0.32s | new: audio 0.96s video 0.33s | so far old 1m 02.9s vs new 12.66s


...

Processing clips (old + new):  36%|███▌      | 440/1225 [1:47:45<4:42:20, 21.58s/it]

[2026-09-14 00:50:37] clip 440/1225 | VOD pos 2.51h | old: audio 19.63s video 0.34s | new: audio 0.97s video 0.36s | so far old 1h 37m 59.4s vs new 9m 45.0s


Processing clips (old + new):  36%|███▌      | 443/1225 [1:48:58<3:12:21, 14.76s/it]

[2026-09-14 00:51:50] ■ process_clips [old + new, interleaved]: 1h 48m 59.2s (interrupted)
[2026-09-14 00:51:50] ■ pipeline_total: 2h 09m 53.0s (interrupted)


KeyboardInterrupt: 

## Analysis: old vs new on identical clips

Reads `clips.csv` from Drive, so it also works after a runtime restart once the setup cell has run with `RESUME_LOG_DIR` set.

In [10]:
import pandas as pd

df = pd.read_csv(CLIPS_CSV)
measured = df[df["skipped"].astype(str) != "True"].drop_duplicates("clip_id", keep="last").sort_values("start_time_s").copy()
if len(measured) < 5:
    raise RuntimeError(
        f"Only {len(measured)} fully measured clips in {CLIPS_CSV}; need at least 5. If clips were skipped "
        "because they already existed, set RESUME_LOG_DIR to the earlier run's log folder, or delete "
        f"{OUTPUT_FOLDER} and {OUTPUT_FOLDER_NEW} clip folders to re-measure.")

n = len(measured)
stride = ENV["config"]["CLIP_STRIDE"]
STEPS = ["target_s", "video_s", "audio_s", "total_s"]
k = max(1, n // 10)
first, last = measured.head(k), measured.tail(k)
pos_h = measured["start_time_h"].to_numpy()


def audio_growth(variant):
    # Seconds of extra audio time per hour further into the VOD (straight-line fit).
    y = measured[f"{variant}_audio_s"].to_numpy()
    if np.ptp(pos_h) == 0:
        return float("nan"), float("nan")
    slope, intercept = np.polyfit(pos_h, y, 1)
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2 = 1 - ((y - (slope * pos_h + intercept)) ** 2).sum() / ss_tot if ss_tot > 0 else float("nan")
    return float(slope), float(r2)


def window_offset(variant):
    # How far the frames actually read ended from the clip's intended window end.
    return (measured[f"{variant}_video_source_end_s"] - (measured["start_time_s"] + CLIP_DURATION)).abs().median()


variants = {}
for v in ("old", "new"):
    slope, r2 = audio_growth(v)
    sums = {step: float(measured[f"{v}_{step}"].sum()) for step in STEPS}
    variants[v] = {
        "measured_totals_s": {step: round(val, 2) for step, val in sums.items()},
        "mean_per_clip_s": {step: round(float(measured[f"{v}_{step}"].mean()), 3) for step in STEPS},
        "audio_share_pct": round(100 * sums["audio_s"] / sums["total_s"], 1),
        "audio_growth_s_per_vod_hour": round(slope, 3),
        "audio_growth_r2": round(r2, 4),
        "first_decile_per_clip_s": {"total": round(float(first[f"{v}_total_s"].mean()), 3),
                                    "audio": round(float(first[f"{v}_audio_s"].mean()), 3)},
        "last_decile_per_clip_s": {"total": round(float(last[f"{v}_total_s"].mean()), 3),
                                   "audio": round(float(last[f"{v}_audio_s"].mean()), 3)},
        "video_window_offset_median_s": round(float(window_offset(v)), 2),
        "clip_stage_at_stride_1_s": round(sums["total_s"] * stride, 1),
    }

old, new = variants["old"], variants["new"]
speedup = {step: round(old["measured_totals_s"][step] / new["measured_totals_s"][step], 2)
           for step in STEPS if new["measured_totals_s"][step] >= 0.5}  # skip steps too small to ratio meaningfully
SUMMARY["head_to_head"] = {
    "clips_measured": n,
    "clip_stride": stride,
    "vod_span_h": [round(float(pos_h.min()), 2), round(float(pos_h.max()), 2)],
    "first_decile_mean_pos_h": round(float(first["start_time_h"].mean()), 2),
    "last_decile_mean_pos_h": round(float(last["start_time_h"].mean()), 2),
    "old": old,
    "new": new,
    "speedup_x": speedup,
    "audio_speedup_last_decile_x": round(old["last_decile_per_clip_s"]["audio"] / new["last_decile_per_clip_s"]["audio"], 1),
    "old_implied_decode_speed_x_realtime": round(3600 / old["audio_growth_s_per_vod_hour"], 1)
                                           if old["audio_growth_s_per_vod_hour"] > 0 else None,
}
write_json(SUMMARY_JSON, SUMMARY)

measured["vod_hour"] = measured["start_time_h"].astype(int)
by_hour = measured.groupby("vod_hour").agg(
    clips=("clip_id", "count"),
    old_audio_s=("old_audio_s", "mean"), new_audio_s=("new_audio_s", "mean"),
    old_video_s=("old_video_s", "mean"), new_video_s=("new_video_s", "mean"),
    old_total_s=("old_total_s", "mean"), new_total_s=("new_total_s", "mean"),
).round(3)
by_hour["total_speedup_x"] = (by_hour["old_total_s"] / by_hour["new_total_s"]).round(1)
by_hour.to_csv(LOG_DIR / "per_vod_hour.csv")

h2h = SUMMARY["head_to_head"]
print(f"Head-to-head on {n} identical clips, VOD positions {h2h['vod_span_h'][0]}h to {h2h['vod_span_h'][1]}h "
      f"(stride {stride})\n")
print(f"{'step':<8}{'old':>16}{'new':>16}{'speedup':>10}")
for step in STEPS:
    print(f"{step.replace('_s', ''):<8}{fmt_duration(old['measured_totals_s'][step]):>16}"
          f"{fmt_duration(new['measured_totals_s'][step]):>16}{(f"{speedup[step]}×" if step in speedup else "—"):>10}")
if stride > 1:
    print(f"\nScaled to every clip: old {fmt_duration(old['clip_stage_at_stride_1_s'])} vs "
          f"new {fmt_duration(new['clip_stage_at_stride_1_s'])}")

print(f"\nAudio share of clip time: old {old['audio_share_pct']}%, new {new['audio_share_pct']}%")
print(f"Audio growth per hour into the VOD: old +{old['audio_growth_s_per_vod_hour']:.2f}s (R² {old['audio_growth_r2']:.3f}), "
      f"new +{new['audio_growth_s_per_vod_hour']:.3f}s")
print(f"Per clip, first 10% (≈{h2h['first_decile_mean_pos_h']}h): old {old['first_decile_per_clip_s']['total']:.2f}s, "
      f"new {new['first_decile_per_clip_s']['total']:.2f}s")
print(f"Per clip, last 10%  (≈{h2h['last_decile_mean_pos_h']}h): old {old['last_decile_per_clip_s']['total']:.2f}s, "
      f"new {new['last_decile_per_clip_s']['total']:.2f}s")
print(f"Video frames' distance from intended window (median): old {old['video_window_offset_median_s']}s, "
      f"new {new['video_window_offset_median_s']}s\n")
display(by_hour)

Head-to-head on 443 identical clips, VOD positions 0.61h to 2.53h (stride 1)

step                 old             new   speedup
target             2.11s           1.70s     1.24×
video           2m 30.1s        2m 39.8s     0.94×
audio       1h 36m 29.9s        7m 07.5s    13.54×
total       1h 39m 02.1s        9m 49.0s    10.09×

Audio share of clip time: old 97.4%, new 72.6%
Audio growth per hour into the VOD: old +7.87s (R² 0.982), new +0.032s
Per clip, first 10% (≈0.7h): old 6.78s, new 1.28s
Per clip, last 10%  (≈2.42h): old 20.43s, new 1.35s
Video frames' distance from intended window (median): old 2288.79s, new 0.04s



,clips,old_audio_s,new_audio_s,old_video_s,new_video_s,old_total_s,new_total_s,total_speedup_x
vod_hour,,,,,,,,
0,91,7.190,0.932,0.331,0.350,7.524,1.285,5.9
1,239,12.699,0.973,0.336,0.361,13.042,1.338,9.7
2,113,18.588,0.976,0.351,0.370,18.942,1.349,14.0


## Export: old vs new archive

The original export zipped the entire output folder, `full_vod.mp4` included, then sent it through a browser download. The new export zips only the triplets and the emote mapping. Each zips its own method's output. The browser download and the Drive copy are skipped because they depend on network speed, but archive sizes are recorded.

In [12]:
def folder_bytes(folder):
    return sum(p.stat().st_size for p in Path(folder).rglob("*") if p.is_file())


if RUN_EXPORT_BENCHMARK:
    free = shutil.disk_usage("/content").free
    needed = folder_bytes(OUTPUT_FOLDER) * 1.1
    if free < needed:
        raise RuntimeError(f"Need ~{needed / 1e9:.1f} GB free for the export benchmark, have {free / 1e9:.1f} GB.")

    # --- Old export (verbatim, minus files.download) ---
    zip_filename = f"dataset_{VOD_ID}_clips"
    with stage("export_old_whole_folder") as old_e:
        shutil.make_archive(zip_filename, 'zip', os.path.dirname(OUTPUT_FOLDER), os.path.basename(OUTPUT_FOLDER))
        old_e["archive_gb"] = round(os.path.getsize(f"{zip_filename}.zip") / 1e9, 3)
    os.remove(f"{zip_filename}.zip")

    # --- New export (verbatim, minus the Drive copy) ---
    zip_path = f"dataset_{VOD_ID}_new_clips.zip"
    with stage("export_new_triplets_only") as new_e:
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            for subdir in ['video', 'audio', 'target']:
                subdir_path = os.path.join(OUTPUT_FOLDER_NEW, subdir)
                if os.path.exists(subdir_path):
                    for root, _, files in os.walk(subdir_path):
                        for file in files:
                            filepath = os.path.join(root, file)
                            zf.write(filepath, os.path.relpath(filepath, os.path.dirname(OUTPUT_FOLDER_NEW)))
            mapping = os.path.join(OUTPUT_FOLDER_NEW, 'emote_mapping.json')
            if os.path.exists(mapping):
                zf.write(mapping, os.path.relpath(mapping, os.path.dirname(OUTPUT_FOLDER_NEW)))
        new_e["archive_gb"] = round(os.path.getsize(zip_path) / 1e9, 3)
    os.remove(zip_path)

    SUMMARY["export"] = {
        "old_archive_gb": old_e["archive_gb"], "old_zip_s": old_e["seconds"],
        "new_archive_gb": new_e["archive_gb"], "new_zip_s": new_e["seconds"],
        "size_reduction_x": round(old_e["archive_gb"] / new_e["archive_gb"], 1) if new_e["archive_gb"] else None,
        "zip_speedup_x": round(old_e["seconds"] / new_e["seconds"], 1) if new_e["seconds"] else None,
        "note": "Old export also required a browser download of old_archive_gb; not timed.",
    }
    write_json(SUMMARY_JSON, SUMMARY)
    e = SUMMARY["export"]
    print(f"Old export: {e['old_archive_gb']} GB archive, zipped in {old_e['human']} (then downloaded through the browser)")
    print(f"New export: {e['new_archive_gb']} GB archive, zipped in {new_e['human']}")
    print(f"→ {e['size_reduction_x']}× smaller, {e['zip_speedup_x']}× faster to zip")

[2026-09-14 00:52:29] ▶ export_old_whole_folder
[2026-09-14 00:53:57] ■ export_old_whole_folder: 1m 28.7s (ok)
[2026-09-14 00:53:58] ▶ export_new_triplets_only
[2026-09-14 00:54:11] ■ export_new_triplets_only: 13.28s (ok)
Old export: 2.14 GB archive, zipped in 1m 28.7s (then downloaded through the browser)
New export: 0.298 GB archive, zipped in 13.28s
→ 7.2× smaller, 6.7× faster to zip


## Headline numbers

Collects the results into plain statements and writes them to `summary.json`. Every number is measured and traces back to `clips.csv` or `stages.json` in the log folder.

In [13]:
headlines = []
h2h, e = SUMMARY.get("head_to_head"), SUMMARY.get("export")

if h2h:
    old, new, sx = h2h["old"], h2h["new"], h2h["speedup_x"]
    scope = f"{h2h['clips_measured']} identical clips"
    if h2h["clip_stride"] > 1:
        scope += f" (every {h2h['clip_stride']}th clip; totals below scale ×{h2h['clip_stride']} to the full VOD)"
    headlines.append(
        f"Clip processing on {scope}: {fmt_duration(old['measured_totals_s']['total_s'])} old → "
        f"{fmt_duration(new['measured_totals_s']['total_s'])} new ({sx['total_s']}× faster).")
    headlines.append(
        f"Audio extraction: {fmt_duration(old['measured_totals_s']['audio_s'])} → "
        f"{fmt_duration(new['measured_totals_s']['audio_s'])} ({sx['audio_s']}× faster); it was "
        f"{old['audio_share_pct']}% of the old pipeline's clip time.")
    headlines.append(
        f"Old per-clip time grew from {old['first_decile_per_clip_s']['total']:.1f}s (≈{h2h['first_decile_mean_pos_h']}h into the VOD) "
        f"to {old['last_decile_per_clip_s']['total']:.1f}s (≈{h2h['last_decile_mean_pos_h']}h); new stayed at "
        f"{new['first_decile_per_clip_s']['total']:.2f}s → {new['last_decile_per_clip_s']['total']:.2f}s.")
    headlines.append(
        f"Late in the VOD, audio per clip was {old['last_decile_per_clip_s']['audio']:.1f}s old vs "
        f"{new['last_decile_per_clip_s']['audio']:.2f}s new ({h2h['audio_speedup_last_decile_x']}× faster).")
    headlines.append(
        f"Old video frames came a median {old['video_window_offset_median_s']}s from the intended window; "
        f"new: {new['video_window_offset_median_s']}s.")
if e:
    headlines.append(f"Export archive: {e['old_archive_gb']} GB → {e['new_archive_gb']} GB "
                     f"({e['size_reduction_x']}× smaller, {e['zip_speedup_x']}× faster to zip).")

SUMMARY["headlines"] = headlines
SUMMARY["caveats"] = [
    "Both methods share the new downloader (yt-dlp + GQL); TwitchDownloaderCLI itself is not measured.",
    "Both methods process the old parser's clip list, so clip counts match the old pipeline, not the new one.",
    "Old and new video times are similar, but the old step never seeks, so its frames come from the wrong part of the VOD.",
    "Clips are interleaved with alternating order so neither method always gets a warmer disk cache.",
    f"Single run on {ENV['cpu_count']} CPUs; Colab hardware varies between sessions.",
]
write_json(SUMMARY_JSON, SUMMARY)

print("HEADLINES")
for h in headlines:
    print(" •", h)
print("\nCAVEATS")
for c in SUMMARY["caveats"]:
    print(" •", c)
print(f"\nAll logs: {LOG_DIR}")

HEADLINES
 • Clip processing on 443 identical clips: 1h 39m 02.1s old → 9m 49.0s new (10.09× faster).
 • Audio extraction: 1h 36m 29.9s → 7m 07.5s (13.54× faster); it was 97.4% of the old pipeline's clip time.
 • Old per-clip time grew from 6.8s (≈0.7h into the VOD) to 20.4s (≈2.42h); new stayed at 1.28s → 1.35s.
 • Late in the VOD, audio per clip was 20.1s old vs 0.98s new (20.5× faster).
 • Old video frames came a median 2288.79s from the intended window; new: 0.04s.
 • Export archive: 2.14 GB → 0.298 GB (7.2× smaller, 6.7× faster to zip).

CAVEATS
 • Both methods share the new downloader (yt-dlp + GQL); TwitchDownloaderCLI itself is not measured.
 • Both methods process the old parser's clip list, so clip counts match the old pipeline, not the new one.
 • Old and new video times are similar, but the old step never seeks, so its frames come from the wrong part of the VOD.
 • Clips are interleaved with alternating order so neither method always gets a warmer disk cache.
 • Single run 